<a href="https://colab.research.google.com/github/msrehman786/IBM-AI-Certification/blob/main/Chatbot_Integrating_your_Chatbot_into_a_Web_Interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!python3 -m pip install flask
!python3 -m pip install flask_cors

Setting up the server

In [ ]:
from flask import Flask
from flask_cors import CORS		# newly added

app = Flask(__name__)
CORS(app)				# newly added

@app.route('/')
def home():
    return 'Hello, World!'

if __name__ == '__main__':
    app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Integrating your chatbot into your Flask server

In [ ]:
#!python3 -m pip install transformers==4.41.2
#!python3 -m pip install torch==2.11.0
#!python3 -m pip install accelerate==0.30.1
#!python3 -m pip install numpy==1.26.4

!pip install transformers pillow torch torchvision torchaudio accelerate numpy

#!python3 -m pip install transformers
#!python3 -m pip install torch
#!python3 -m pip install accelerate
#!python3 -m pip install numpy


copy the code to initialize your chatbot

In [ ]:
!pip install flask pyngrok flask_cors

In [ ]:
!which ngrok

/usr/local/bin/ngrok


In [ ]:
!ngrok authtoken '3IBE5FhsNRF52VxuQwh0JnlsAqu_6RoqvbJKa7EwEAespyiP'

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
!git clone https://github.com/ibm-developer-skills-network/LLM_application_chatbot


fatal: destination path 'LLM_application_chatbot' already exists and is not an empty directory.


In [ ]:
!python3 -m pip install -r LLM_application_chatbot/requirements.txt

In [ ]:
from flask import Flask, request, render_template
from flask_cors import CORS
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from pyngrok import ngrok

app = Flask(__name__)
CORS(app)

# Optional: Set ngrok auth token (skip if not using one)
ngrok.set_auth_token("3IBE5FhsNRF52VxuQwh0JnlsAqu_6RoqvbJKa7EwEAespyiP")  # Replace with your token

# Open a ngrok tunnel to port 5000 (where Flask will run)
public_url = ngrok.connect(5001).public_url
print(f"✅ Flask app is live at: {public_url}")

model_name = "facebook/blenderbot-400M-distill"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
conversation_history = []

# Define routes (same as before, plus any new ones)
#@app.route('/')
#def home():
#    return "Hello, Flask in Google Colab!"

@app.route('/', methods=['GET'])
def home():
    return render_template('index.html')

@app.route('/chatbot', methods=['POST'])
def handle_prompt():
    # Read prompt from HTTP request body
    data = request.get_json()
    input_text = data["prompt"]

    # Keep only the recent conversation
    conversation_history[:] = conversation_history[-6:]

    # Create conversation history string
    history = "\n".join(conversation_history)
    prompt = history + f"\nUser: {input_text}\nBot:"

    # Tokenize the input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate the response
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
        repetition_penalty=1.3,
        do_sample=True,
        temperature=0.6,
        top_p=0.85
    )

    # Decode the response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    # Add interaction to conversation history
    conversation_history.append(f"User: {input_text}")
    conversation_history.append(f"Bot: {response}")

    return response

app.run(host='0.0.0.0', port=5001)

#if __name__ == '__main__':
    #app.run()
    # Run the Flask app on the Colab server’s public interface (0.0.0.0) and port 5000
#    app.run(host='0.0.0.0', port=5001)


✅ Flask app is live at: https://snowless-persuader-nest.ngrok-free.dev


Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 14:02:50] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 14:02:50] "GET /static/css/style.css HTTP/1.1" 304 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 14:02:51] "GET /static/script.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 14:03:05] "GET /static/user.jpeg HTTP/1.1" 304 -
[transformers] Both `max_new_tokens` (=60) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:werkzeug:127.0.0.1 - - [01/Sep/2026 14:03:21] "POST /chatbot HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - -

In [ ]:
!curl -X POST -H "Content-Type: application/json" -d '{"prompt": "Hello, how are you today?"}' https://snowless-persuader-nest.ngrok-free.dev/chatbot